# Options from Zero · Episode 5: The one-step tree: copy the option

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The tree · 4. How many shares · 5. How much to borrow · 6. Check both states · 7. Delta for other strikes · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 5,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

If interest rates rose, would the \$105 call get dearer or cheaper compared with the put? Parity: call − put = share − $K e^{-rT}$, and a higher rate shrinks $K e^{-rT}$, so the call gains on the put.

In [ ]:
gap = lambda rate: black_scholes(S, K, T, rate, vol, True) - black_scholes(S, K, T, rate, vol, False)
out["answer"] = {"rate_hi_pct": 5.0, "gap_now": gap(r), "gap_hi": gap(0.05), "parity_hi": S - K * math.exp(-0.05 * T)}
out["answer"]

## 3. The tree

Episode 3's toy world: today \$100; in a year \$120 or \$80. The call struck at \$105 pays \$15 or nothing.

In [ ]:
UP, DOWN = 120.0, 80.0
tree = one_step(S, UP, DOWN, K, T, r)
out["tree"] = {"up": UP, "down": DOWN, "pay_up": tree["v_up"], "pay_down": tree["v_down"], "price": tree["price"],
               "nodes": [{"t": 0, "k": 0, "price": S, "value": tree["price"]},
                         {"t": 1, "k": 0, "price": DOWN, "value": tree["v_down"]},
                         {"t": 1, "k": 1, "price": UP, "value": tree["v_up"]}]}

## 4. How many shares

Hold $\Delta$ shares. Between the two outcomes the share's value moves by $\Delta \times (120 - 80)$ and the option's by $15 - 0$. Make them move together:

$$\Delta = \frac{C_u - C_d}{S_u - S_d} = \frac{15 - 0}{120 - 80}$$

In [ ]:
out["delta"] = {"delta": tree["delta"], "payoff_spread": tree["v_up"] - tree["v_down"], "share_spread": UP - DOWN,
                "shares_cost": tree["delta"] * S}
out["delta"]

## 5. How much to borrow

In the down state the shares are worth $\Delta \times 80 = 30$ but the option pays nothing, so borrow today exactly what repays \$30 in a year: $30 e^{-rT}$.

In [ ]:
borrow = -tree["cash"]
out["borrow"] = {"borrow": borrow, "repay": borrow * math.exp(r * T), "price": tree["delta"] * S - borrow}
out["borrow"]

## 6. Check both states

The copy (0.375 shares, minus the loan) pays what the call pays in both states, so today it must cost what the call costs.

In [ ]:
rows = []
for name, s_T, pay in (("share at $120", UP, tree["v_up"]), ("share at $80", DOWN, tree["v_down"])):
    shares_value = tree["delta"] * s_T
    rows.append({"state": name, "shares": shares_value, "repay": -borrow * math.exp(r * T),
                 "copy": shares_value - borrow * math.exp(r * T), "call": pay})
assert all(abs(x["copy"] - x["call"]) < 1e-12 for x in rows)
out["check"] = {"rows": rows}
pd.DataFrame(rows)

**Copying a put.** The same recipe copies the \$105 put, which pays nothing up and \$25 down. Its delta is negative: *sell* shares and *lend* cash. The result matches put-call parity from Episode 4.

In [ ]:
put = one_step(S, UP, DOWN, K, T, r, call=False)
out["put"] = {"pay_up": put["v_up"], "pay_down": put["v_down"], "delta": put["delta"], "delta_abs": abs(put["delta"]),
              "short_value": abs(put["delta"]) * S, "lend": put["cash"], "price": put["price"],
              "parity_lhs": tree["price"] - put["price"], "parity_rhs": S - K * math.exp(-r * T)}
assert abs(out["put"]["parity_lhs"] - out["put"]["parity_rhs"]) < 1e-12
out["put"]

## 7. Delta for other strikes

Delta is the number of shares that copies (and so hedges) the option. A strike of \$80 pays \$40 up and nothing down, exactly the share's own \$40 swing, so its delta is 1: it moves one for one with the share. A strike of \$120 or more never pays: delta 0.

In [ ]:
rows = [{"strike": k, **{key: one_step(S, UP, DOWN, k, T, r)[key] for key in ("delta", "price")}}
        for k in (80.0, 90.0, 100.0, 105.0, 110.0, 120.0)]
out["strikes"] = {"rows": rows, "rows_shown": rows[1:], "delta_80": rows[0]["delta"], "delta_120": rows[-1]["delta"], "delta_120_strike": rows[-1]["strike"]}
pd.DataFrame(rows)

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")